<a href="https://colab.research.google.com/github/korkutanapa/DCASE2025_TASK2/blob/main/ORJ_DCASE_FEATURE_SELECTION_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
"""
JOINT FEATURE-SUBSET + k + DISTANCE-METRIC TDA ORACLE SEARCH
=============================================================

Purpose
-------
For every labeled DCASE development machine, jointly search:

    * TDA feature subset S
    * k in {3, 5, 10, 20, 30}
    * distance metric in:
        - L1 / Manhattan
        - L2 / Euclidean
        - L4 / Minkowski p=4
        - Linf / Chebyshev
        - regularized Mahalanobis

Normal-reference kNN anomaly score
----------------------------------
For a test sample x and a selected feature subset S:

    score(x; S, k, metric)
        = mean distance from x to its k nearest NORMAL train samples

IMPORTANT SCORE DIRECTION
-------------------------
Larger distance = more anomalous.
The code NEVER flips the anomaly score and NEVER replaces AUC by max(AUC, 1-AUC).

Study type
----------
This is a SUPERVISED DEVELOPMENT / ORACLE capability study.
Development test labels are intentionally used to select feature subset, k,
and distance metric. Therefore the final results are optimistic development
upper bounds and are NOT directly deployable unseen-machine selection rules.

Preprocessing
-------------
For each machine:
    * only NORMAL train samples form the reference bank
    * median imputation is fitted on normal train only
    * StandardScaler is fitted on normal train only
    * test data are transformed with train-fitted preprocessing

Search strategy
---------------
1 feature:
    exhaustive over every usable TDA feature for every (metric, k)

2 features:
    exhaustive over every feature pair for every (metric, k)

3+ features:
    configuration-preserving multi-beam forward search
    * strongest candidates are retained separately for every (metric, k)
    * their feature subsets are unioned before expanding to the next size
    * this allows the optimal feature set to depend on the distance metric

Efficiency
----------
For each subset:
    * one ordinary cKDTree is built for L1/L2/L4/Linf
    * that same tree is queried under the four Minkowski norms
    * for each metric, MAX_K=30 neighbors are queried once
    * k=3/5/10/20/30 scores are derived by cumulative means

Mahalanobis
-----------
Mahalanobis distance uses the NORMAL-train covariance of the selected subset.
The covariance is regularized by shrinkage toward a scaled identity matrix:

    Sigma_reg = (1-lambda) Sigma + lambda * mu * I + jitter * I
    mu = trace(Sigma) / d

Then the data are linearly transformed so Euclidean distance in the transformed
space equals the regularized Mahalanobis distance in the original subset space.

Default objective
-----------------
DCASE-oriented score:

    HM(AUC_source, AUC_target, pAUC@0.1)

Tie-breakers:
    pAUC@0.1
    min(AUC_source, AUC_target)
    AUC_all
    fewer features
    smaller k
    simpler metric only as a final exact-tie preference

Set OBJECTIVE = "auc_all" if overall AUC should be the primary oracle target.

Main outputs
------------
/content/joint_k_distance_machine_specific_tda_oracle/
    joint_k_distance_machine_specific_tda_oracle.xlsx
    best_oracle_feature_pool.json

Excel sheets
------------
    Best_Overall
        single best (subset, k, metric) per machine

    Best_By_Metric
        best subset+k for each machine and metric

    Best_By_K
        best subset+metric for each machine and k

    Best_By_K_Metric
        best subset for each machine x k x metric

    Best_By_Size_K_Metric
        best subset for every machine x subset-size x k x metric

    Top_Singles
    Top_Pairs
    Beam_History
    Expansion_Pool
    Feature_Stability
    Metric_Usage
    Config

Usage in Google Colab
---------------------
Place the machine train/test Excel files somewhere under /content and run:

    %run /content/joint_k_distance_machine_specific_tda_oracle.py

Expected filename pattern:
    cubical_mel_tda_features_dev_<MACHINE>_train.xlsx
    cubical_mel_tda_features_dev_<MACHINE>_test.xlsx

Duplicate downloaded files such as test(4).xlsx are also recognized; the newest
matching file for each machine/split is used.
"""

from __future__ import annotations

from pathlib import Path
from itertools import combinations, count
from collections import Counter
import heapq
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")


# =============================================================================
# CONFIGURATION
# =============================================================================

DATA_DIR = Path("/content")

K_VALUES = (3, 5, 10, 20, 30)
MAX_K = max(K_VALUES)
PAUC_MAX_FPR = 0.10

# Distance metrics searched jointly with feature subset and k.
# cKDTree accepts p=1, 2, 4 and np.inf for the four Minkowski-family metrics.
LP_METRICS = {
    "L1_Manhattan": 1.0,
    "L2_Euclidean": 2.0,
    "L4_Minkowski": 4.0,
    "Linf_Chebyshev": np.inf,
}
USE_MAHALANOBIS = True
MAHALANOBIS_NAME = "Mahalanobis_reg"

# Covariance shrinkage for Mahalanobis.
# 0.0 = sample covariance only, 1.0 = scaled identity only.
MAHALANOBIS_SHRINKAGE = 0.10
MAHALANOBIS_JITTER = 1e-6

# "dcase_hm" = HM(AUC_source, AUC_target, pAUC@0.1)
# "auc_all"  = overall AUC first
OBJECTIVE = "dcase_hm"

# Search subset sizes 1..MAX_FEATURES.
MAX_FEATURES = 20

# Higher-order search settings.
# IMPORTANT: this is PER (metric, k), not just per k.
BEAM_PER_CONFIG = 6
SAVE_TOP_PER_SIZE_PER_CONFIG = 6

# Expansion pool is built from strong exhaustive singles/pairs across ALL configs.
TOP_SINGLE_PER_CONFIG_FOR_POOL = 30
TOP_PAIR_PER_CONFIG_FOR_POOL = 80
MAX_EXPANSION_POOL = 100

# Set True for a stronger but much slower 3+ search.
EXPAND_WITH_ALL_VALID_FEATURES = False

# Train-normal redundancy control for higher-order subsets.
USE_REDUNDANCY_FILTER = True
MAX_ABS_TRAIN_CORR = 0.995

# Optional local same-size swap refinement for each machine x k x metric.
ENABLE_FINAL_SWAP_REFINEMENT = True
SWAP_CANDIDATE_LIMIT = 50
MAX_SWAP_PASSES = 1

# Full pair space is evaluated, but only top rows are kept in memory per config.
PAIR_KEEP_TOP_PER_CONFIG = max(
    250,
    TOP_PAIR_PER_CONFIG_FOR_POOL,
    BEAM_PER_CONFIG,
)

# cKDTree query workers; -1 means all supported CPU threads.
KD_WORKERS = -1
PAIR_PROGRESS_EVERY = 5000

OUTPUT_DIR = Path("/content/joint_k_distance_machine_specific_tda_oracle")
OUTPUT_XLSX = OUTPUT_DIR / "joint_k_distance_machine_specific_tda_oracle.xlsx"
OUTPUT_JSON = OUTPUT_DIR / "best_oracle_feature_pool.json"

EXCLUDED_TDA_FEATURES = {
    "H0_points_raw",
    "H1_points_raw",
}

# Exact-tie preference only. It has NO effect unless all performance and size/k
# tie-breakers are identical.
METRIC_TIE_ORDER = {
    "L2_Euclidean": 0,
    "L1_Manhattan": 1,
    "L4_Minkowski": 2,
    "Linf_Chebyshev": 3,
    MAHALANOBIS_NAME: 4,
}


def all_metric_names() -> tuple[str, ...]:
    names = list(LP_METRICS)
    if USE_MAHALANOBIS:
        names.append(MAHALANOBIS_NAME)
    return tuple(names)


METRIC_NAMES = all_metric_names()
CONFIG_KEYS = tuple((metric, k) for metric in METRIC_NAMES for k in K_VALUES)


# =============================================================================
# DATASET DETECTION
# =============================================================================

FILE_PATTERN = re.compile(
    r"^cubical_mel_tda_features_dev_(?P<machine>.+?)_"
    r"(?P<split>train|test)(?:\(\d+\))?$",
    flags=re.IGNORECASE,
)


def detect_datasets(data_dir: Path) -> dict:
    """Find newest complete train/test Excel pair for every machine."""

    records = []

    for path in data_dir.rglob("*.xlsx"):
        match = FILE_PATTERN.match(path.stem)
        if not match:
            continue

        records.append(
            {
                "machine_raw": match.group("machine"),
                "machine_key": match.group("machine").lower(),
                "split": match.group("split").lower(),
                "path": path,
                "mtime": path.stat().st_mtime,
            }
        )

    if not records:
        raise FileNotFoundError(
            f"No matching DCASE development Excel files were found under {data_dir}."
        )

    selected = {}
    for rec in records:
        key = (rec["machine_key"], rec["split"])
        old = selected.get(key)
        if old is None or rec["mtime"] > old["mtime"]:
            selected[key] = rec

    datasets = {}
    for machine_key in sorted({key[0] for key in selected}):
        train_rec = selected.get((machine_key, "train"))
        test_rec = selected.get((machine_key, "test"))

        if train_rec is None or test_rec is None:
            print(
                f"WARNING: incomplete pair ignored for {machine_key}: "
                f"train={train_rec is not None}, test={test_rec is not None}"
            )
            continue

        datasets[machine_key] = {
            "display_name": train_rec["machine_raw"],
            "train": train_rec["path"],
            "test": test_rec["path"],
        }

    if not datasets:
        raise FileNotFoundError(
            "No complete train/test machine pairs were found under /content."
        )

    return datasets


# =============================================================================
# LABEL AND DOMAIN HELPERS
# =============================================================================


def labels_to_binary(series: pd.Series) -> np.ndarray:
    """Convert labels to 0=normal, 1=anomaly."""

    if pd.api.types.is_numeric_dtype(series):
        values = pd.to_numeric(series, errors="coerce")
        if values.isna().any():
            raise ValueError("Some numeric labels could not be interpreted.")
        return (values > 0).astype(np.int8).to_numpy()

    text = series.astype(str).str.strip().str.lower()
    anomaly_words = {"1", "anomaly", "anomalous", "abnormal", "fault", "faulty", "ng"}
    normal_words = {"0", "normal", "healthy", "ok"}

    output = []
    unknown = set()

    for value in text:
        if (
            value in anomaly_words
            or "anomal" in value
            or "abnormal" in value
            or "fault" in value
        ):
            output.append(1)
        elif value in normal_words or "normal" in value or "healthy" in value:
            output.append(0)
        else:
            output.append(-1)
            unknown.add(value)

    if unknown:
        raise ValueError(f"Unknown labels: {sorted(unknown)}")

    return np.asarray(output, dtype=np.int8)


def _normalize_domain_value(value) -> str:
    text = str(value).strip().lower()
    if "source" in text:
        return "source"
    if "target" in text:
        return "target"
    return "unknown"


def infer_test_domains(test_df: pd.DataFrame) -> np.ndarray:
    """
    Infer source/target domain from explicit metadata or file_id/file_path text.
    """

    explicit_candidates = [
        "domain",
        "domain_label",
        "source_target",
        "data_domain",
        "dataset_domain",
    ]

    lower_to_real = {str(c).lower(): c for c in test_df.columns}

    for candidate in explicit_candidates:
        if candidate in lower_to_real:
            col = lower_to_real[candidate]
            domains = test_df[col].map(_normalize_domain_value).to_numpy()
            if np.mean(domains != "unknown") >= 0.95:
                return domains

    text_columns = []
    for candidate in ["file_id", "file_path", "filename", "path", "wav_path"]:
        if candidate in lower_to_real:
            text_columns.append(lower_to_real[candidate])

    if not text_columns:
        return np.asarray(["unknown"] * len(test_df), dtype=object)

    combined = test_df[text_columns[0]].astype(str)
    for col in text_columns[1:]:
        combined = combined + " " + test_df[col].astype(str)

    return combined.map(_normalize_domain_value).to_numpy()


# =============================================================================
# METRICS AND RANKING
# =============================================================================


def safe_auc(y: np.ndarray, scores: np.ndarray, max_fpr=None) -> float:
    if len(y) == 0 or len(np.unique(y)) < 2:
        return float("nan")
    return float(roc_auc_score(y, scores, max_fpr=max_fpr))


def harmonic_mean(values) -> float:
    values = np.asarray(values, dtype=float)
    if len(values) == 0 or np.any(~np.isfinite(values)) or np.any(values <= 0):
        return float("nan")
    return float(len(values) / np.sum(1.0 / values))


def compute_metrics(
    y_test: np.ndarray,
    domains: np.ndarray,
    scores: np.ndarray,
) -> dict:
    auc_all = safe_auc(y_test, scores)
    pauc_01 = safe_auc(y_test, scores, max_fpr=PAUC_MAX_FPR)

    source_mask = domains == "source"
    target_mask = domains == "target"

    auc_source = safe_auc(y_test[source_mask], scores[source_mask])
    auc_target = safe_auc(y_test[target_mask], scores[target_mask])

    dcase_hm = harmonic_mean([auc_source, auc_target, pauc_01])

    if np.isfinite(auc_source) and np.isfinite(auc_target):
        min_domain_auc = float(min(auc_source, auc_target))
    else:
        min_domain_auc = float("nan")

    return {
        "auc_all": auc_all,
        "auc_source": auc_source,
        "auc_target": auc_target,
        "pauc_01": pauc_01,
        "dcase_hm": dcase_hm,
        "min_domain_auc": min_domain_auc,
    }


def _finite_or_neg_inf(value) -> float:
    try:
        value = float(value)
    except Exception:
        return -math.inf
    return value if np.isfinite(value) else -math.inf


def ranking_tuple(row: dict) -> tuple:
    """Higher tuple = better."""

    metric_tie = -METRIC_TIE_ORDER.get(row.get("distance_metric", ""), 999)

    if OBJECTIVE == "dcase_hm":
        return (
            _finite_or_neg_inf(row["dcase_hm"]),
            _finite_or_neg_inf(row["pauc_01"]),
            _finite_or_neg_inf(row["min_domain_auc"]),
            _finite_or_neg_inf(row["auc_all"]),
            -int(row["n_features"]),
            -int(row["k"]),
            metric_tie,
        )

    if OBJECTIVE == "auc_all":
        return (
            _finite_or_neg_inf(row["auc_all"]),
            _finite_or_neg_inf(row["pauc_01"]),
            _finite_or_neg_inf(row["dcase_hm"]),
            -int(row["n_features"]),
            -int(row["k"]),
            metric_tie,
        )

    raise ValueError(f"Unknown OBJECTIVE={OBJECTIVE!r}")


def canonical_subset(features) -> tuple:
    return tuple(sorted(set(features)))


def subset_to_text(subset) -> str:
    return " | ".join(subset)


# =============================================================================
# TRAIN-ONLY PREPROCESSING
# =============================================================================


def prepare_machine_data(train_df: pd.DataFrame, test_df: pd.DataFrame) -> dict:
    """
    Prepare one machine using NORMAL train data only for preprocessing/reference.
    """

    if "label" not in test_df.columns:
        raise ValueError("Test file does not contain a 'label' column.")

    # Keep only normal train samples if a train label exists.
    if "label" in train_df.columns:
        try:
            y_train = labels_to_binary(train_df["label"])
            if np.any(y_train == 1):
                print(
                    f"  WARNING: removing {int(np.sum(y_train == 1))} "
                    "non-normal rows from train reference bank."
                )
                train_df = train_df.loc[y_train == 0].reset_index(drop=True)
        except Exception:
            pass

    features = [
        col
        for col in train_df.columns
        if (
            col in test_df.columns
            and str(col).startswith(("H0_", "H1_"))
            and col not in EXCLUDED_TDA_FEATURES
        )
    ]

    if len(features) < 2:
        raise ValueError("Fewer than two common TDA feature columns were found.")

    train_num = (
        train_df[features]
        .replace([np.inf, -np.inf], np.nan)
        .apply(pd.to_numeric, errors="coerce")
    )
    test_num = (
        test_df[features]
        .replace([np.inf, -np.inf], np.nan)
        .apply(pd.to_numeric, errors="coerce")
    )

    nonempty = [f for f in features if not train_num[f].isna().all()]
    train_num = train_num[nonempty]
    test_num = test_num[nonempty]

    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(train_num)
    X_test = imputer.transform(test_num)

    std = np.std(X_train, axis=0)
    valid_mask = np.isfinite(std) & (std > 1e-12)

    feature_names = [
        feature for feature, keep in zip(nonempty, valid_mask) if keep
    ]

    X_train = X_train[:, valid_mask]
    X_test = X_test[:, valid_mask]

    if len(feature_names) < 2:
        raise ValueError("Fewer than two nonconstant TDA features remain.")

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float64)
    X_test = scaler.transform(X_test).astype(np.float64)

    y_test = labels_to_binary(test_df["label"])
    if len(np.unique(y_test)) < 2:
        raise ValueError("Test labels must contain both normal and anomaly samples.")

    domains = infer_test_domains(test_df)

    if OBJECTIVE == "dcase_hm":
        for domain_name in ("source", "target"):
            mask = domains == domain_name
            if np.sum(mask) == 0 or len(np.unique(y_test[mask])) < 2:
                raise ValueError(
                    f"OBJECTIVE='dcase_hm' requires normal+anomaly test samples "
                    f"for domain={domain_name!r}."
                )

    if len(X_train) < MAX_K:
        raise ValueError(
            f"At least {MAX_K} normal train samples are required; found {len(X_train)}."
        )

    feature_to_index = {f: i for i, f in enumerate(feature_names)}

    corr = np.corrcoef(X_train, rowvar=False).astype(np.float64)
    corr[~np.isfinite(corr)] = 0.0

    return {
        "X_train": X_train,
        "X_test": X_test,
        "y_test": y_test,
        "domains": domains,
        "feature_names": feature_names,
        "feature_to_index": feature_to_index,
        "corr": corr,
    }


# =============================================================================
# DISTANCE HELPERS
# =============================================================================


def query_knn_distances(
    tree: cKDTree,
    X_test: np.ndarray,
    k: int,
    p: float = 2.0,
) -> np.ndarray:
    """cKDTree query wrapper with p-norm and scipy-version compatibility."""

    try:
        distances, _ = tree.query(X_test, k=k, p=p, workers=KD_WORKERS)
    except TypeError:
        distances, _ = tree.query(X_test, k=k, p=p)

    distances = np.asarray(distances, dtype=np.float64)
    if distances.ndim == 1:
        distances = distances[:, None]

    return distances


def mahalanobis_linear_transform(
    X_train: np.ndarray,
    X_test: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Convert regularized Mahalanobis distance into Euclidean distance.

    If Sigma_reg is the regularized covariance and VI=inv(Sigma_reg), find
    L such that VI = L L^T. Then:

        d_M(x,y)^2 = (x-y)^T VI (x-y)
                     = || (x-y) L ||_2^2

    so cKDTree can be used on X @ L.
    """

    d = X_train.shape[1]

    if d == 1:
        variance = float(np.var(X_train[:, 0], ddof=1))
        if not np.isfinite(variance) or variance <= 0:
            variance = 1.0
        sigma_reg = np.array([[variance + MAHALANOBIS_JITTER]], dtype=np.float64)
    else:
        cov = np.cov(X_train, rowvar=False, ddof=1)
        cov = np.asarray(cov, dtype=np.float64)

        if cov.ndim == 0:
            cov = np.array([[float(cov)]], dtype=np.float64)

        cov[~np.isfinite(cov)] = 0.0

        mu = float(np.trace(cov) / d)
        if not np.isfinite(mu) or mu <= 0:
            mu = 1.0

        identity = np.eye(d, dtype=np.float64)
        sigma_reg = (
            (1.0 - MAHALANOBIS_SHRINKAGE) * cov
            + MAHALANOBIS_SHRINKAGE * mu * identity
            + MAHALANOBIS_JITTER * identity
        )

    # Symmetrize to remove tiny numerical asymmetry.
    sigma_reg = 0.5 * (sigma_reg + sigma_reg.T)

    # Stable eigendecomposition; avoids failure on nearly singular covariance.
    eigvals, eigvecs = np.linalg.eigh(sigma_reg)
    floor = max(MAHALANOBIS_JITTER, 1e-12)
    eigvals = np.clip(eigvals, floor, None)

    # VI^(1/2) = Q diag(1/sqrt(lambda)) Q^T.
    whitener = (eigvecs * (1.0 / np.sqrt(eigvals))) @ eigvecs.T

    X_train_m = X_train @ whitener
    X_test_m = X_test @ whitener

    return X_train_m, X_test_m


def metric_scores_from_distances(distances: np.ndarray) -> dict[int, np.ndarray]:
    """Return mean kNN distances for all K_VALUES from one MAX_K query."""

    cumulative = np.cumsum(distances, axis=1)
    return {
        int(k): cumulative[:, k - 1] / float(k)
        for k in K_VALUES
    }


# =============================================================================
# SUBSET EVALUATION FOR ALL METRICS AND ALL k
# =============================================================================


def evaluate_subset_all_configs(
    machine_name: str,
    subset,
    data: dict,
) -> list[dict]:
    """
    Evaluate one subset for every requested distance metric and k.

    IMPORTANT:
        high mean kNN distance = anomaly
        no score sign reversal
        no AUC flipping
    """

    subset = canonical_subset(subset)
    index_map = data["feature_to_index"]

    if not subset or not all(feature in index_map for feature in subset):
        return []

    indices = [index_map[feature] for feature in subset]
    X_train = data["X_train"][:, indices]
    X_test = data["X_test"][:, indices]

    rows = []

    # -------------------------------------------------------------------------
    # L1, L2, L4, Linf: one tree, different p values.
    # -------------------------------------------------------------------------
    tree = cKDTree(X_train)

    for metric_name, p_value in LP_METRICS.items():
        distances = query_knn_distances(
            tree,
            X_test,
            MAX_K,
            p=p_value,
        )

        scores_by_k = metric_scores_from_distances(distances)

        for k, scores in scores_by_k.items():
            metrics = compute_metrics(
                data["y_test"],
                data["domains"],
                scores,
            )

            rows.append(
                {
                    "machine": machine_name,
                    "distance_metric": metric_name,
                    "metric_p": "inf" if np.isinf(p_value) else float(p_value),
                    "k": int(k),
                    "n_features": len(subset),
                    "subset": subset,
                    "features": subset_to_text(subset),
                    **metrics,
                }
            )

    # -------------------------------------------------------------------------
    # Regularized Mahalanobis.
    # -------------------------------------------------------------------------
    if USE_MAHALANOBIS:
        X_train_m, X_test_m = mahalanobis_linear_transform(X_train, X_test)
        tree_m = cKDTree(X_train_m)
        distances_m = query_knn_distances(
            tree_m,
            X_test_m,
            MAX_K,
            p=2.0,
        )

        scores_by_k = metric_scores_from_distances(distances_m)

        for k, scores in scores_by_k.items():
            metrics = compute_metrics(
                data["y_test"],
                data["domains"],
                scores,
            )

            rows.append(
                {
                    "machine": machine_name,
                    "distance_metric": MAHALANOBIS_NAME,
                    "metric_p": "mahalanobis",
                    "k": int(k),
                    "n_features": len(subset),
                    "subset": subset,
                    "features": subset_to_text(subset),
                    **metrics,
                }
            )

    return rows


# =============================================================================
# REDUNDANCY FILTER
# =============================================================================


def subset_is_redundant(subset, data: dict) -> bool:
    if not USE_REDUNDANCY_FILTER or len(subset) < 2:
        return False

    idx = [data["feature_to_index"][f] for f in subset]
    subcorr = np.abs(data["corr"][np.ix_(idx, idx)])
    upper = subcorr[np.triu_indices(len(idx), k=1)]

    return bool(np.any(upper >= MAX_ABS_TRAIN_CORR))


# =============================================================================
# TOP-N HEAP HELPERS
# =============================================================================


_heap_counter = count()


def push_top(heap: list, row: dict, limit: int):
    item = (ranking_tuple(row), next(_heap_counter), row)

    if len(heap) < limit:
        heapq.heappush(heap, item)
    elif item[0] > heap[0][0]:
        heapq.heapreplace(heap, item)


def sorted_heap_rows(heap: list) -> list[dict]:
    return sorted(
        [item[2] for item in heap],
        key=ranking_tuple,
        reverse=True,
    )


def best_row(rows: list[dict]) -> dict:
    if not rows:
        raise ValueError("No rows available for best-row selection.")
    return max(rows, key=ranking_tuple)


# =============================================================================
# EXHAUSTIVE SINGLE SEARCH
# =============================================================================


def exhaustive_single_search(machine_name: str, data: dict) -> list[dict]:
    print("  Exhaustive single-feature search over all metrics and k ...")

    rows = []
    features = data["feature_names"]

    for i, feature in enumerate(features, start=1):
        rows.extend(
            evaluate_subset_all_configs(machine_name, (feature,), data)
        )

        if i % 25 == 0 or i == len(features):
            print(f"    singles {i}/{len(features)}")

    return rows


# =============================================================================
# EXHAUSTIVE PAIR SEARCH
# =============================================================================


def exhaustive_pair_search(
    machine_name: str,
    data: dict,
) -> dict[tuple[str, int], list[dict]]:
    """
    Evaluate every feature pair for all metric x k configurations.

    Only strongest rows per configuration are retained in memory/output.
    """

    features = data["feature_names"]
    n_pairs = len(features) * (len(features) - 1) // 2

    print(
        f"  Exhaustive pair search: {n_pairs:,} pairs x "
        f"{len(METRIC_NAMES)} metrics x {len(K_VALUES)} k values ..."
    )

    heaps = {config: [] for config in CONFIG_KEYS}

    for pair_no, pair in enumerate(combinations(features, 2), start=1):
        rows = evaluate_subset_all_configs(machine_name, pair, data)

        for row in rows:
            key = (row["distance_metric"], row["k"])
            push_top(heaps[key], row, PAIR_KEEP_TOP_PER_CONFIG)

        if pair_no % PAIR_PROGRESS_EVERY == 0 or pair_no == n_pairs:
            print(f"    pairs {pair_no:,}/{n_pairs:,}")

    return {
        config: sorted_heap_rows(heaps[config])
        for config in CONFIG_KEYS
    }


# =============================================================================
# EXPANSION POOL
# =============================================================================


def build_expansion_pool(
    single_rows: list[dict],
    pair_top_by_config: dict[tuple[str, int], list[dict]],
    valid_features: list[str],
) -> list[str]:
    """Build a configuration-balanced feature expansion pool."""

    ordered = []

    def add(feature):
        if feature in valid_features and feature not in ordered:
            ordered.append(feature)

    single_by_config = {}
    for metric, k in CONFIG_KEYS:
        rows_cfg = [
            row
            for row in single_rows
            if row["distance_metric"] == metric and row["k"] == k
        ]
        single_by_config[(metric, k)] = sorted(
            rows_cfg,
            key=ranking_tuple,
            reverse=True,
        )

    # Round-robin strongest singles across all metric x k configs.
    for rank_idx in range(TOP_SINGLE_PER_CONFIG_FOR_POOL):
        for config in CONFIG_KEYS:
            ranked = single_by_config[config]
            if rank_idx < len(ranked):
                add(ranked[rank_idx]["subset"][0])

                if (
                    not EXPAND_WITH_ALL_VALID_FEATURES
                    and MAX_EXPANSION_POOL is not None
                    and len(ordered) >= MAX_EXPANSION_POOL
                ):
                    break
        if (
            not EXPAND_WITH_ALL_VALID_FEATURES
            and MAX_EXPANSION_POOL is not None
            and len(ordered) >= MAX_EXPANSION_POOL
        ):
            break

    # Round-robin strongest pairs across all metric x k configs.
    if (
        EXPAND_WITH_ALL_VALID_FEATURES
        or MAX_EXPANSION_POOL is None
        or len(ordered) < MAX_EXPANSION_POOL
    ):
        for rank_idx in range(TOP_PAIR_PER_CONFIG_FOR_POOL):
            for config in CONFIG_KEYS:
                ranked = pair_top_by_config[config]
                if rank_idx < len(ranked):
                    for feature in ranked[rank_idx]["subset"]:
                        add(feature)

                    if (
                        not EXPAND_WITH_ALL_VALID_FEATURES
                        and MAX_EXPANSION_POOL is not None
                        and len(ordered) >= MAX_EXPANSION_POOL
                    ):
                        break

            if (
                not EXPAND_WITH_ALL_VALID_FEATURES
                and MAX_EXPANSION_POOL is not None
                and len(ordered) >= MAX_EXPANSION_POOL
            ):
                break

    if EXPAND_WITH_ALL_VALID_FEATURES:
        for feature in valid_features:
            add(feature)
    elif MAX_EXPANSION_POOL is not None:
        ordered = ordered[:MAX_EXPANSION_POOL]

    return ordered


# =============================================================================
# MULTI-FEATURE CONFIGURATION-PRESERVING BEAM SEARCH
# =============================================================================


def multi_feature_search(
    machine_name: str,
    data: dict,
    single_rows: list[dict],
    pair_top_by_config: dict[tuple[str, int], list[dict]],
    expansion_pool: list[str],
) -> tuple[list[dict], list[dict]]:
    """
    Search sizes 3..MAX_FEATURES while preserving top candidates separately
    for every (distance metric, k) configuration.
    """

    best_by_size_config = []
    beam_history = []

    # Exact size-1 best for each configuration.
    for metric, k in CONFIG_KEYS:
        rows_cfg = [
            row
            for row in single_rows
            if row["distance_metric"] == metric and row["k"] == k
        ]
        ranked = sorted(rows_cfg, key=ranking_tuple, reverse=True)
        if not ranked:
            continue

        row = ranked[0].copy()
        row["search_stage"] = "exhaustive_single"
        best_by_size_config.append(row)

    # Exact size-2 best + initial beam union.
    beam_subsets = set()

    for config in CONFIG_KEYS:
        metric, k = config
        ranked = pair_top_by_config[config]
        if not ranked:
            continue

        row = ranked[0].copy()
        row["search_stage"] = "exhaustive_pair"
        best_by_size_config.append(row)

        for rank, candidate in enumerate(ranked[:BEAM_PER_CONFIG], start=1):
            beam_subsets.add(candidate["subset"])

            hist = candidate.copy()
            hist["search_stage"] = "exhaustive_pair_seed"
            hist["beam_rank_for_config"] = rank
            beam_history.append(hist)

    # Sizes 3+.
    for target_size in range(3, MAX_FEATURES + 1):
        print(
            f"  Multi search size={target_size}: "
            f"parents={len(beam_subsets)}, pool={len(expansion_pool)}"
        )

        candidate_subsets = set()

        for base_subset in beam_subsets:
            base_set = set(base_subset)

            for feature in expansion_pool:
                if feature in base_set:
                    continue

                candidate = canonical_subset((*base_subset, feature))
                if len(candidate) != target_size:
                    continue

                if subset_is_redundant(candidate, data):
                    continue

                candidate_subsets.add(candidate)

        if not candidate_subsets:
            print(f"    no valid candidates at size={target_size}; stopping.")
            break

        keep_n = max(BEAM_PER_CONFIG, SAVE_TOP_PER_SIZE_PER_CONFIG)
        heaps = {config: [] for config in CONFIG_KEYS}

        total = len(candidate_subsets)
        progress_every = max(500, total // 10)

        for candidate_no, subset in enumerate(candidate_subsets, start=1):
            rows = evaluate_subset_all_configs(machine_name, subset, data)

            for row in rows:
                config = (row["distance_metric"], row["k"])
                push_top(heaps[config], row, keep_n)

            if candidate_no % progress_every == 0 or candidate_no == total:
                print(f"    candidates {candidate_no:,}/{total:,}")

        new_beam_subsets = set()

        for config in CONFIG_KEYS:
            metric, k = config
            ranked = sorted_heap_rows(heaps[config])
            if not ranked:
                continue

            best = ranked[0].copy()
            best["search_stage"] = "multi_config_beam"
            best_by_size_config.append(best)

            print(
                f"    {metric:<16s} k={k:2d}  "
                f"HM={best['dcase_hm']:.4f}  "
                f"AUC={best['auc_all']:.4f}  "
                f"src={best['auc_source']:.4f}  "
                f"tgt={best['auc_target']:.4f}  "
                f"pAUC={best['pauc_01']:.4f}"
            )

            for row in ranked[:BEAM_PER_CONFIG]:
                new_beam_subsets.add(row["subset"])

            for rank, row in enumerate(
                ranked[:SAVE_TOP_PER_SIZE_PER_CONFIG],
                start=1,
            ):
                hist = row.copy()
                hist["search_stage"] = "multi_config_beam"
                hist["beam_rank_for_config"] = rank
                beam_history.append(hist)

        if not new_beam_subsets:
            break

        beam_subsets = new_beam_subsets

    return best_by_size_config, beam_history


# =============================================================================
# FINAL SAME-SIZE SWAP REFINEMENT
# =============================================================================


def refine_best_for_config(
    machine_name: str,
    start_row: dict,
    data: dict,
    expansion_pool: list[str],
) -> dict:
    """Greedy same-size swap refinement for one fixed (metric, k)."""

    if not ENABLE_FINAL_SWAP_REFINEMENT or start_row["n_features"] <= 2:
        return start_row

    target_metric = start_row["distance_metric"]
    target_k = int(start_row["k"])
    current = start_row.copy()

    local_cache = {}
    candidates_pool = expansion_pool[:SWAP_CANDIDATE_LIMIT]

    def evaluate_target(subset):
        subset = canonical_subset(subset)
        if subset not in local_cache:
            rows = evaluate_subset_all_configs(machine_name, subset, data)
            local_cache[subset] = {
                (row["distance_metric"], row["k"]): row
                for row in rows
            }
        return local_cache[subset].get((target_metric, target_k))

    for _pass in range(MAX_SWAP_PASSES):
        improved = False
        best_candidate = current
        current_subset = current["subset"]
        current_set = set(current_subset)

        for remove_feature in current_subset:
            reduced = current_set - {remove_feature}

            for add_feature in candidates_pool:
                if add_feature in reduced:
                    continue

                candidate_subset = canonical_subset((*reduced, add_feature))
                if len(candidate_subset) != len(current_subset):
                    continue

                if subset_is_redundant(candidate_subset, data):
                    continue

                row = evaluate_target(candidate_subset)
                if row is None:
                    continue

                if ranking_tuple(row) > ranking_tuple(best_candidate):
                    best_candidate = row
                    improved = True

        if not improved:
            break

        current = best_candidate.copy()

    current["search_stage"] = "final_swap_refinement"
    return current


# =============================================================================
# MACHINE SEARCH
# =============================================================================


def search_one_machine(machine_name: str, data: dict) -> dict:
    print("\n" + "=" * 110)
    print(f"MACHINE: {machine_name}")
    print("=" * 110)
    print(f"  usable TDA features   = {len(data['feature_names'])}")
    print(f"  normal train samples  = {len(data['X_train'])}")
    print(f"  labeled test samples  = {len(data['X_test'])}")
    print(f"  distance metrics      = {METRIC_NAMES}")
    print(f"  k values              = {K_VALUES}")
    print(
        "  test domains          = "
        + ", ".join(
            f"{d}:{int(np.sum(data['domains'] == d))}"
            for d in ["source", "target", "unknown"]
            if np.sum(data["domains"] == d) > 0
        )
    )

    # 1) Exact singles.
    single_rows = exhaustive_single_search(machine_name, data)

    # 2) Exact pairs.
    pair_top_by_config = exhaustive_pair_search(machine_name, data)

    # 3) Higher-order expansion pool.
    expansion_pool = build_expansion_pool(
        single_rows,
        pair_top_by_config,
        data["feature_names"],
    )
    print(f"  expansion pool size = {len(expansion_pool)}")

    # 4) Metric+k-preserving higher-order search.
    best_by_size_config, beam_history = multi_feature_search(
        machine_name,
        data,
        single_rows,
        pair_top_by_config,
        expansion_pool,
    )

    # 5) Best pre-refinement subset for every metric x k.
    pre_best_by_config = []
    for metric, k in CONFIG_KEYS:
        candidates = [
            row
            for row in best_by_size_config
            if row["distance_metric"] == metric and row["k"] == k
        ]
        if candidates:
            pre_best_by_config.append(best_row(candidates).copy())

    # 6) Same-size local refinement for every metric x k.
    for row in pre_best_by_config:
        refined = refine_best_for_config(
            machine_name,
            row,
            data,
            expansion_pool,
        )
        if ranking_tuple(refined) > ranking_tuple(row):
            best_by_size_config.append(refined)

    # 7) Final best subset for every metric x k.
    best_by_config = []
    for metric, k in CONFIG_KEYS:
        candidates = [
            row
            for row in best_by_size_config
            if row["distance_metric"] == metric and row["k"] == k
        ]
        if candidates:
            best_by_config.append(best_row(candidates).copy())

    # 8) Best k/subset for each metric.
    best_by_metric = []
    for metric in METRIC_NAMES:
        candidates = [
            row for row in best_by_config
            if row["distance_metric"] == metric
        ]
        if candidates:
            best_by_metric.append(best_row(candidates).copy())

    # 9) Best metric/subset for each k.
    best_by_k = []
    for k in K_VALUES:
        candidates = [row for row in best_by_config if row["k"] == k]
        if candidates:
            best_by_k.append(best_row(candidates).copy())

    # 10) Single best subset+k+metric for the machine.
    best_overall = best_row(best_by_config).copy()

    pair_top_rows = []
    for config in CONFIG_KEYS:
        pair_top_rows.extend(pair_top_by_config[config])

    return {
        "single_rows": single_rows,
        "pair_top_rows": pair_top_rows,
        "best_by_size_config": best_by_size_config,
        "beam_history": beam_history,
        "expansion_pool": expansion_pool,
        "best_by_config": best_by_config,
        "best_by_metric": best_by_metric,
        "best_by_k": best_by_k,
        "best_overall": best_overall,
    }


# =============================================================================
# EXPORT HELPERS
# =============================================================================


def rows_to_dataframe(rows: list[dict]) -> pd.DataFrame:
    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    if "subset" in df.columns:
        df = df.drop(columns=["subset"])

    preferred = [
        "machine",
        "distance_metric",
        "metric_p",
        "k",
        "n_features",
        "features",
        "dcase_hm",
        "auc_all",
        "auc_source",
        "auc_target",
        "pauc_01",
        "min_domain_auc",
        "search_stage",
        "beam_rank_for_config",
        "rank_for_machine_config",
    ]

    cols = [c for c in preferred if c in df.columns] + [
        c for c in df.columns if c not in preferred
    ]

    return df[cols]


def select_best_group_rows(rows: list[dict], group_keys: list[str]) -> list[dict]:
    groups = {}

    for row in rows:
        key = tuple(row[k] for k in group_keys)
        previous = groups.get(key)
        if previous is None or ranking_tuple(row) > ranking_tuple(previous):
            groups[key] = row

    return list(groups.values())


def make_feature_stability(best_by_config_rows: list[dict]) -> pd.DataFrame:
    config_count = Counter()
    machine_sets = {}
    metric_sets = {}
    k_sets = {}

    for row in best_by_config_rows:
        machine = row["machine"]
        metric = row["distance_metric"]
        k = int(row["k"])

        for feature in row["subset"]:
            config_count[feature] += 1
            machine_sets.setdefault(feature, set()).add(machine)
            metric_sets.setdefault(feature, set()).add(metric)
            k_sets.setdefault(feature, set()).add(k)

    rows = []
    for feature in sorted(config_count):
        rows.append(
            {
                "feature": feature,
                "selected_machine_metric_k_count": config_count[feature],
                "selected_machine_count": len(machine_sets[feature]),
                "selected_metric_count": len(metric_sets[feature]),
                "selected_k_count": len(k_sets[feature]),
                "machines": " | ".join(sorted(machine_sets[feature])),
                "distance_metrics": " | ".join(sorted(metric_sets[feature])),
                "k_values": " | ".join(map(str, sorted(k_sets[feature]))),
            }
        )

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows).sort_values(
        [
            "selected_machine_count",
            "selected_machine_metric_k_count",
            "selected_metric_count",
            "selected_k_count",
            "feature",
        ],
        ascending=[False, False, False, False, True],
    )


def make_metric_usage(
    best_overall_rows: list[dict],
    best_by_metric_rows: list[dict],
) -> pd.DataFrame:
    """Small summary showing which metric wins and metric-specific best scores."""

    overall_wins = Counter(row["distance_metric"] for row in best_overall_rows)

    rows = []
    for metric in METRIC_NAMES:
        metric_rows = [
            row for row in best_by_metric_rows
            if row["distance_metric"] == metric
        ]

        hm_vals = [row["dcase_hm"] for row in metric_rows if np.isfinite(row["dcase_hm"])]
        auc_vals = [row["auc_all"] for row in metric_rows if np.isfinite(row["auc_all"])]
        pauc_vals = [row["pauc_01"] for row in metric_rows if np.isfinite(row["pauc_01"])]

        rows.append(
            {
                "distance_metric": metric,
                "overall_machine_win_count": int(overall_wins.get(metric, 0)),
                "mean_best_dcase_hm": float(np.mean(hm_vals)) if hm_vals else np.nan,
                "mean_best_auc_all": float(np.mean(auc_vals)) if auc_vals else np.nan,
                "mean_best_pauc_01": float(np.mean(pauc_vals)) if pauc_vals else np.nan,
            }
        )

    return pd.DataFrame(rows).sort_values(
        ["overall_machine_win_count", "mean_best_dcase_hm"],
        ascending=[False, False],
    )


def _json_float(value):
    value = float(value)
    return value if np.isfinite(value) else None


def save_json_feature_pool(
    best_by_config_rows: list[dict],
    best_overall_rows: list[dict],
    output_path: Path,
):
    feature_pool_all_configs = sorted(
        {
            feature
            for row in best_by_config_rows
            for feature in row["subset"]
        }
    )

    feature_pool_overall = sorted(
        {
            feature
            for row in best_overall_rows
            for feature in row["subset"]
        }
    )

    feature_counts = Counter(
        feature
        for row in best_by_config_rows
        for feature in row["subset"]
    )

    best_config_nested = {}
    for row in best_by_config_rows:
        machine = row["machine"]
        metric = row["distance_metric"]
        k = str(row["k"])

        best_config_nested.setdefault(machine, {}).setdefault(metric, {})[k] = {
            "n_features": int(row["n_features"]),
            "features": list(row["subset"]),
            "dcase_hm": _json_float(row["dcase_hm"]),
            "auc_all": _json_float(row["auc_all"]),
            "auc_source": _json_float(row["auc_source"]),
            "auc_target": _json_float(row["auc_target"]),
            "pauc_01": _json_float(row["pauc_01"]),
        }

    overall_results = {}
    for row in best_overall_rows:
        overall_results[row["machine"]] = {
            "best_distance_metric": row["distance_metric"],
            "best_k": int(row["k"]),
            "n_features": int(row["n_features"]),
            "features": list(row["subset"]),
            "dcase_hm": _json_float(row["dcase_hm"]),
            "auc_all": _json_float(row["auc_all"]),
            "auc_source": _json_float(row["auc_source"]),
            "auc_target": _json_float(row["auc_target"]),
            "pauc_01": _json_float(row["pauc_01"]),
        }

    payload = {
        "description": (
            "Machine-specific supervised development oracle jointly selecting "
            "TDA feature subset, k, and distance metric."
        ),
        "warning": (
            "Development test labels are used for oracle selection. This is not "
            "a deployable unseen-machine selection rule."
        ),
        "score_direction": (
            "Higher mean kNN distance = more anomalous; scores are never inverted."
        ),
        "k_values": list(K_VALUES),
        "distance_metrics": list(METRIC_NAMES),
        "mahalanobis_shrinkage": MAHALANOBIS_SHRINKAGE if USE_MAHALANOBIS else None,
        "objective": OBJECTIVE,
        "objective_definition": (
            "HM(AUC_source, AUC_target, pAUC@0.1), then pAUC, "
            "min(source,target AUC), AUC_all"
            if OBJECTIVE == "dcase_hm"
            else "AUC_all, then pAUC"
        ),
        "n_unique_features_in_best_machine_metric_k_solutions": len(
            feature_pool_all_configs
        ),
        "feature_pool_best_machine_metric_k": feature_pool_all_configs,
        "feature_selection_count_across_machine_metric_k": dict(
            sorted(feature_counts.items(), key=lambda x: (-x[1], x[0]))
        ),
        "n_unique_features_in_best_overall_machine_solutions": len(
            feature_pool_overall
        ),
        "feature_pool_best_overall": feature_pool_overall,
        "best_by_machine_metric_k": best_config_nested,
        "best_overall_by_machine": overall_results,
    }

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)


# =============================================================================
# MAIN
# =============================================================================


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("=" * 110)
    print("JOINT FEATURE SUBSET + k + DISTANCE METRIC TDA ORACLE SEARCH")
    print("=" * 110)
    print(f"DATA_DIR                  : {DATA_DIR}")
    print(f"K_VALUES                  : {K_VALUES}")
    print(f"DISTANCE_METRICS          : {METRIC_NAMES}")
    print(f"OBJECTIVE                 : {OBJECTIVE}")
    print(f"MAX_FEATURES              : {MAX_FEATURES}")
    print(f"BEAM_PER_CONFIG           : {BEAM_PER_CONFIG}")
    print(f"MAX_EXPANSION_POOL        : {MAX_EXPANSION_POOL}")
    print(f"MAHALANOBIS_SHRINKAGE     : {MAHALANOBIS_SHRINKAGE}")
    print(f"OUTPUT_DIR                : {OUTPUT_DIR}")
    print()
    print("IMPORTANT: labeled development test data selects subset + k + metric.")
    print("Anomaly score direction: larger mean kNN distance = more anomalous.")
    print("No score inversion and no AUC flipping are used.")

    datasets = detect_datasets(DATA_DIR)

    print("\nDetected complete machine pairs:")
    for key in sorted(datasets):
        info = datasets[key]
        print(
            f"  {info['display_name']:<15s} "
            f"train={info['train'].name}  test={info['test'].name}"
        )

    all_single_rows = []
    all_pair_top_rows = []
    all_best_by_size_config_rows = []
    all_beam_history_rows = []
    all_best_by_config_rows = []
    all_best_by_metric_rows = []
    all_best_by_k_rows = []
    all_best_overall_rows = []
    expansion_pool_rows = []

    for machine_no, machine_key in enumerate(sorted(datasets), start=1):
        info = datasets[machine_key]
        machine_name = info["display_name"]

        print(f"\n[{machine_no}/{len(datasets)}] Loading {machine_name} ...")

        train_df = pd.read_excel(info["train"])
        test_df = pd.read_excel(info["test"])

        data = prepare_machine_data(train_df, test_df)
        result = search_one_machine(machine_name, data)

        all_single_rows.extend(result["single_rows"])
        all_pair_top_rows.extend(result["pair_top_rows"])
        all_best_by_size_config_rows.extend(result["best_by_size_config"])
        all_beam_history_rows.extend(result["beam_history"])
        all_best_by_config_rows.extend(result["best_by_config"])
        all_best_by_metric_rows.extend(result["best_by_metric"])
        all_best_by_k_rows.extend(result["best_by_k"])
        all_best_overall_rows.append(result["best_overall"])

        for pool_order, feature in enumerate(result["expansion_pool"], start=1):
            expansion_pool_rows.append(
                {
                    "machine": machine_name,
                    "pool_order": pool_order,
                    "feature": feature,
                }
            )

        print("\n  BEST BY DISTANCE METRIC")
        for row in sorted(
            result["best_by_metric"],
            key=lambda r: METRIC_TIE_ORDER.get(r["distance_metric"], 999),
        ):
            print(
                f"    {row['distance_metric']:<16s}  "
                f"k={row['k']:2d}  n={row['n_features']:2d}  "
                f"HM={row['dcase_hm']:.4f}  AUC={row['auc_all']:.4f}  "
                f"src={row['auc_source']:.4f}  tgt={row['auc_target']:.4f}  "
                f"pAUC={row['pauc_01']:.4f}"
            )

        print("\n  BEST BY k (metric free)")
        for row in sorted(result["best_by_k"], key=lambda r: r["k"]):
            print(
                f"    k={row['k']:2d}  {row['distance_metric']:<16s}  "
                f"n={row['n_features']:2d}  HM={row['dcase_hm']:.4f}  "
                f"AUC={row['auc_all']:.4f}"
            )

        best = result["best_overall"]
        print("\n  OVERALL BEST subset + k + metric")
        print(
            f"    metric={best['distance_metric']}  k={best['k']}  "
            f"n={best['n_features']}  HM={best['dcase_hm']:.4f}  "
            f"AUC={best['auc_all']:.4f}"
        )
        print(f"    {best['features']}")

    # -------------------------------------------------------------------------
    # Final grouped results after refinements.
    # -------------------------------------------------------------------------

    best_by_size_config_final = select_best_group_rows(
        all_best_by_size_config_rows,
        ["machine", "distance_metric", "k", "n_features"],
    )

    best_by_config_final = select_best_group_rows(
        best_by_size_config_final,
        ["machine", "distance_metric", "k"],
    )

    best_by_metric_final = select_best_group_rows(
        best_by_config_final,
        ["machine", "distance_metric"],
    )

    best_by_k_final = select_best_group_rows(
        best_by_config_final,
        ["machine", "k"],
    )

    best_overall_final = select_best_group_rows(
        best_by_config_final,
        ["machine"],
    )

    # -------------------------------------------------------------------------
    # Top singles export.
    # -------------------------------------------------------------------------

    top_single_export_rows = []
    single_groups = {}

    for row in all_single_rows:
        key = (row["machine"], row["distance_metric"], row["k"])
        single_groups.setdefault(key, []).append(row)

    for rows in single_groups.values():
        ranked = sorted(rows, key=ranking_tuple, reverse=True)
        for rank, row in enumerate(ranked[:50], start=1):
            r = row.copy()
            r["rank_for_machine_config"] = rank
            top_single_export_rows.append(r)

    # -------------------------------------------------------------------------
    # Top pairs export.
    # -------------------------------------------------------------------------

    top_pair_export_rows = []
    pair_groups = {}

    for row in all_pair_top_rows:
        key = (row["machine"], row["distance_metric"], row["k"])
        pair_groups.setdefault(key, []).append(row)

    for rows in pair_groups.values():
        ranked = sorted(rows, key=ranking_tuple, reverse=True)
        for rank, row in enumerate(ranked, start=1):
            r = row.copy()
            r["rank_for_machine_config"] = rank
            top_pair_export_rows.append(r)

    feature_stability = make_feature_stability(best_by_config_final)
    metric_usage = make_metric_usage(best_overall_final, best_by_metric_final)

    # -------------------------------------------------------------------------
    # DataFrames.
    # -------------------------------------------------------------------------

    best_overall_df = rows_to_dataframe(best_overall_final).sort_values("machine")

    best_by_metric_df = rows_to_dataframe(best_by_metric_final).sort_values(
        ["machine", "distance_metric"]
    )

    best_by_k_df = rows_to_dataframe(best_by_k_final).sort_values(
        ["machine", "k"]
    )

    best_by_config_df = rows_to_dataframe(best_by_config_final).sort_values(
        ["machine", "distance_metric", "k"]
    )

    best_by_size_config_df = rows_to_dataframe(
        best_by_size_config_final
    ).sort_values(
        ["machine", "distance_metric", "k", "n_features"]
    )

    top_singles_df = rows_to_dataframe(top_single_export_rows).sort_values(
        ["machine", "distance_metric", "k", "rank_for_machine_config"]
    )

    top_pairs_df = rows_to_dataframe(top_pair_export_rows).sort_values(
        ["machine", "distance_metric", "k", "rank_for_machine_config"]
    )

    beam_history_df = rows_to_dataframe(all_beam_history_rows).sort_values(
        ["machine", "n_features", "distance_metric", "k", "beam_rank_for_config"]
    )

    expansion_pool_df = pd.DataFrame(expansion_pool_rows).sort_values(
        ["machine", "pool_order"]
    )

    config_df = pd.DataFrame(
        {
            "parameter": [
                "K_VALUES",
                "DISTANCE_METRICS",
                "OBJECTIVE",
                "PAUC_MAX_FPR",
                "MAX_FEATURES",
                "BEAM_PER_CONFIG",
                "SAVE_TOP_PER_SIZE_PER_CONFIG",
                "TOP_SINGLE_PER_CONFIG_FOR_POOL",
                "TOP_PAIR_PER_CONFIG_FOR_POOL",
                "MAX_EXPANSION_POOL",
                "EXPAND_WITH_ALL_VALID_FEATURES",
                "USE_REDUNDANCY_FILTER",
                "MAX_ABS_TRAIN_CORR",
                "ENABLE_FINAL_SWAP_REFINEMENT",
                "SWAP_CANDIDATE_LIMIT",
                "MAX_SWAP_PASSES",
                "MAHALANOBIS_SHRINKAGE",
                "MAHALANOBIS_JITTER",
                "score_direction",
            ],
            "value": [
                str(K_VALUES),
                str(METRIC_NAMES),
                OBJECTIVE,
                PAUC_MAX_FPR,
                MAX_FEATURES,
                BEAM_PER_CONFIG,
                SAVE_TOP_PER_SIZE_PER_CONFIG,
                TOP_SINGLE_PER_CONFIG_FOR_POOL,
                TOP_PAIR_PER_CONFIG_FOR_POOL,
                MAX_EXPANSION_POOL,
                EXPAND_WITH_ALL_VALID_FEATURES,
                USE_REDUNDANCY_FILTER,
                MAX_ABS_TRAIN_CORR,
                ENABLE_FINAL_SWAP_REFINEMENT,
                SWAP_CANDIDATE_LIMIT,
                MAX_SWAP_PASSES,
                MAHALANOBIS_SHRINKAGE,
                MAHALANOBIS_JITTER,
                "higher mean kNN distance = anomaly; no score inversion",
            ],
        }
    )

    # -------------------------------------------------------------------------
    # Excel export.
    # -------------------------------------------------------------------------

    with pd.ExcelWriter(OUTPUT_XLSX) as writer:
        best_overall_df.to_excel(writer, sheet_name="Best_Overall", index=False)
        best_by_metric_df.to_excel(writer, sheet_name="Best_By_Metric", index=False)
        best_by_k_df.to_excel(writer, sheet_name="Best_By_K", index=False)
        best_by_config_df.to_excel(writer, sheet_name="Best_By_K_Metric", index=False)
        best_by_size_config_df.to_excel(
            writer,
            sheet_name="Best_By_Size_K_Metric",
            index=False,
        )
        top_singles_df.to_excel(writer, sheet_name="Top_Singles", index=False)
        top_pairs_df.to_excel(writer, sheet_name="Top_Pairs", index=False)
        beam_history_df.to_excel(writer, sheet_name="Beam_History", index=False)
        expansion_pool_df.to_excel(writer, sheet_name="Expansion_Pool", index=False)
        feature_stability.to_excel(writer, sheet_name="Feature_Stability", index=False)
        metric_usage.to_excel(writer, sheet_name="Metric_Usage", index=False)
        config_df.to_excel(writer, sheet_name="Config", index=False)

    # -------------------------------------------------------------------------
    # JSON feature-pool export.
    # -------------------------------------------------------------------------

    save_json_feature_pool(
        best_by_config_final,
        best_overall_final,
        OUTPUT_JSON,
    )

    # -------------------------------------------------------------------------
    # Final console summaries.
    # -------------------------------------------------------------------------

    print("\n" + "=" * 110)
    print("FINAL BEST subset + k + DISTANCE METRIC PER MACHINE")
    print("=" * 110)

    display_cols = [
        "machine",
        "distance_metric",
        "k",
        "n_features",
        "dcase_hm",
        "auc_all",
        "auc_source",
        "auc_target",
        "pauc_01",
        "features",
    ]

    print(best_overall_df[display_cols].round(4).to_string(index=False))

    print("\n" + "=" * 110)
    print("DISTANCE METRIC USAGE SUMMARY")
    print("=" * 110)
    print(metric_usage.round(4).to_string(index=False))

    print("\nSaved:")
    print(f"  {OUTPUT_XLSX}")
    print(f"  {OUTPUT_JSON}")


if __name__ == "__main__":
    main()